In [ ]:
import os
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
from scipy.stats import ttest_1samp
from scipy.ndimage import gaussian_filter
from typing import Optional, Sequence, Dict, List

import cmocean
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.colors as mcolors
import cartopy.crs as ccrs


In [ ]:
class SurfaceVarDiffProcessor:
    """
    Compute ensemble mean, spread, bias, and significance vs obs for a single date
    and save results into a compact NetCDF file for later plotting.
    """

    def __init__(self, data_dict: Dict, *,
                 variable: str = "PRECT",
                 obs_variable: Optional[str] = None,
                 fscale: Optional[float] = None,
                 obs_fscale: Optional[float] = None,
                 landmask_file: Optional[str] = None,
                 landmask_var: str = "landfrac",
                 mask_land: bool = True,
                 confidence_level: float = 0.05):
        self.data_dict = data_dict
        self.variable = variable
        self.obs_variable = obs_variable or variable
        self.fscale = 1.0 if fscale is None else fscale
        self.obs_fscale = 1.0 if obs_fscale is None else obs_fscale
        self.landmask_var = landmask_var
        self.mask_land = mask_land
        self.confidence_level = float(confidence_level)

        self.landmask = None
        if landmask_file and os.path.exists(landmask_file):
            ds_mask = xr.open_dataset(landmask_file)
            lm = ds_mask[self.landmask_var]
            lon_name = next((k for k in lm.coords if k.lower() in ["lon","longitude"]), None)
            if lon_name:
                lon = lm[lon_name]
                if lon.min() < 0 or lon.max() > 360:
                    new_lon = (lon % 360).sortby(lon)
                    lm = lm.assign_coords({lon_name: new_lon}).sortby(lon_name)
            self.landmask = lm

    # ----------------- helpers -----------------
    def _get_lat_lon_names(self, da: xr.DataArray):
        lon_name = next((k for k in da.coords if k.lower() in ["lon", "longitude"]), None)
        lat_name = next((k for k in da.coords if k.lower() in ["lat", "latitude"]), None)
        return lon_name, lat_name

    def _apply_landmask(self, da: xr.DataArray):
        if self.mask_land and (self.landmask is not None):
            lm = self.landmask
            # interpolate nearest to avoid fill; works for 2D lat/lon grid
            lm_i = lm.interp_like(da, method="nearest")
            return da.where(lm_i < 0.5)
        return da

    def _wrap_longitude_for_plot(self, da: xr.DataArray):
        lon_name, _ = self._get_lat_lon_names(da)
        if lon_name is None:
            return da
        lon = da[lon_name]
        if float(lon.max() - lon.min()) < 359:  # pad to avoid seam
            pad = da.isel({lon_name: 0}).assign_coords({lon_name: lon.isel({lon_name: 0}) + 360})
            da = xr.concat([da, pad], dim=lon_name)
        return da

    def _weighted_stats(self, model: xr.DataArray, obs: xr.DataArray):
        # area weights by latitude
        lat_name = next((k for k in model.coords if k.lower() in ["lat","latitude"]), None)
        w1d = np.cos(np.deg2rad(model[lat_name]))
        w1d = w1d / w1d.mean()
        w2d = xr.ones_like(model) * w1d

        diff = (model - obs)
        rmse = np.sqrt(((diff**2) * w2d).where(np.isfinite(diff)).mean()).item()

        m = (model * w2d).where(np.isfinite(model))
        o = (obs   * w2d).where(np.isfinite(obs))
        mmean = m.mean()
        omean = o.mean()
        cov = ((m - mmean) * (o - omean)).mean()
        stdm = np.sqrt(((m - mmean)**2).mean())
        stdo = np.sqrt(((o - omean)**2).mean())
        pcc = (cov / (stdm * stdo + 1e-12)).item()
        return float(pcc), float(rmse)

    def _load_dataset(self, info: Dict, year: int, ensemble: Optional[str] = None) -> xr.Dataset:
        template = info["template"]
        path = info["path"]
        if "%(" in template:  # dict style
            vals = {"year": year}
            if ensemble is not None:
                vals["ensemble"] = ensemble
            filename = template % vals
        else:
            try:
                filename = template % year
            except Exception:
                filename = template % {"year": year}
        fp = os.path.join(path, filename)
        if not os.path.exists(fp):
            raise FileNotFoundError(f"[ERROR] File not found: {fp}")
        return xr.open_dataset(fp)

    def _load_model_ensemble_snapshot(self, model_info: Dict, target_date: pd.Timestamp) -> xr.DataArray:
        nens = int(model_info["nens"])
        ens_list: List[xr.DataArray] = []
        for n in range(1, nens + 1):
            en = f"EN{n:02d}"
            ds = self._load_dataset(model_info, target_date.year, ensemble=en)
            da = ds[self.variable].sel(time=target_date, method="nearest", tolerance="12h") * self.fscale
            ens_list.append(da.squeeze())
        ens = xr.concat(ens_list, dim="ensemble").assign_coords(ensemble=np.arange(1, nens + 1))
        # NOTE: no auto-chunk here; leave chunking choice to caller or to_disk
        return ens

    def _load_obs_snapshot(self, obs_info: Dict, target_date: pd.Timestamp) -> xr.DataArray:
        ds = self._load_dataset(obs_info, target_date.year)
        obs = ds[self.obs_variable].sel(time=target_date, method="nearest", tolerance="12h") * self.obs_fscale
        # normalize longitudes
        lon_name = next((k for k in obs.coords if k.lower() in ["lon","longitude"]), None)
        if lon_name:
            lon = obs[lon_name]
            if lon.min() < 0 or lon.max() > 360:
                new_lon = (lon % 360).sortby(lon)
                obs = obs.assign_coords({lon_name: new_lon}).sortby(lon_name)
        return self._apply_landmask(obs)

    def _compute_significance_mask(self, ens: xr.DataArray, obs: xr.DataArray, alpha=0.05) -> xr.DataArray:
        if ens.shape[1:] != obs.shape:
            ens = ens.interp_like(obs)
        diff = ens - obs
        # t-test against 0 (population mean 0) along ensemble axis
        tstat, pvals = ttest_1samp(
            diff.transpose("ensemble", ...).values, popmean=0.0, axis=0, nan_policy="omit"
        )
        sig = xr.DataArray((pvals < alpha), coords=obs.coords, dims=obs.dims)
        return sig

    def process_and_save(self, *,
                         model_key: str,
                         obs_key: str = "OBS",
                         target_date: str = "2012-01-01",
                         out_nc: str):
        """
        Compute fields for one model vs one obs on one date; write NetCDF.
        """
        td = pd.to_datetime(target_date)

        model_info = self.data_dict[model_key]
        obs_info = self.data_dict[obs_key]

        ens = self._load_model_ensemble_snapshot(model_info, td)
        obs = self._load_obs_snapshot(obs_info, td)

        if ens.shape[1:] != obs.shape:
            ens = ens.interp_like(obs)

        mean_da = ens.mean("ensemble")
        std_da  = ens.std("ensemble", ddof=1)
        bias    = mean_da - obs

        # apply land mask (obs already masked in loader; but re-apply to keep consistency)
        mean_da = self._apply_landmask(mean_da)
        std_da  = self._apply_landmask(std_da)
        bias    = self._apply_landmask(bias)

        # global stats
        pcc, rmse = self._weighted_stats(mean_da, obs.interp_like(mean_da))

        # significance
        sig = self._compute_significance_mask(ens, obs, alpha=self.confidence_level)
        sig = self._apply_landmask(sig)

        # (Optional) wrap longitudes for plotting convenience; store unwrapped too if desired.
        mean_wr = self._wrap_longitude_for_plot(mean_da)
        std_wr  = self._wrap_longitude_for_plot(std_da)
        bias_wr = self._wrap_longitude_for_plot(bias)
        sig_wr  = self._wrap_longitude_for_plot(sig)
        obs_wr  = self._wrap_longitude_for_plot(obs)

        # Build dataset to save
        ds_out = xr.Dataset(
            {
                "model_mean": mean_wr,
                "model_std":  std_wr,
                "obs_field":  obs_wr,
                "bias":       bias_wr,
                "sig_mask":   sig_wr.astype("i1"),  # 0/1 to save space
            }
        )
        ds_out.attrs.update({
            "variable": self.variable,
            "obs_variable": self.obs_variable,
            "model_key": model_key,
            "obs_key": obs_key,
            "target_date": str(pd.to_datetime(target_date).date()),
            "fscale": float(self.fscale),
            "obs_fscale": float(self.obs_fscale),
            "mask_land": bool(self.mask_land),
            "confidence_level": float(self.confidence_level),
            "pcc_global": pcc,
            "rmse_global": rmse,
        })

        # Lightweight encodings (set zlib if you like)
        enc = {vn: {"zlib": True, "complevel": 4} for vn in ds_out.data_vars}
        ds_out.to_netcdf(out_nc, encoding=enc)
        print(f"[SAVED] {out_nc}  (PCC={pcc:.3f}, RMSE={rmse:.3f})")


In [ ]:
if __name__ == "__main__":
    # Paths
    top_path = "/compyfs/zhan391/v3_dart_cda_scratch"
    out_path = os.path.join(top_path, "diag_dart")
    fig_path = "./figures"
    os.makedirs(out_path, exist_ok=True)
    os.makedirs(fig_path, exist_ok=True)

    landmask_file = os.path.join(out_path, "landmask_1x1.nc")
    landmask_var = "landmask"   # <- name of the var inside landmask_file
    mask_land = False
    confidence_level = 0.05

    # Configuration
    model_keys = ["CTRLEN10", "CAPTEN10", "DARTEN20", "DARTEN40"]
    obs_key = "GPCP"
    variable = "PRECT"
    obs_variable = "PRECT"

    # PRECT: kg m-2 s-1  -> mm/day uses x 86400.0
    fscale = 86400.0
    # GPCP daily is typically already mm/day
    obs_fscale = 1.0

    # Data sources
    data_dict = {
        "GPCP": {
            "path": "/compyfs/zhan391/acme_init/Observations/GPCP/daily",
            "template": "PRECT.daily.%(year)s.nc",
            "frequency": "daily",
            "nens": 1,
        },
        "ERA5": {
            "path": f"{top_path}/Observations/ERA5/6hourly",
            "template": "ERA5.6hourly.en00.TREFHT.%(year)s01-%(year)s12.nc",
            "frequency": "6hourly",
            "nens": 1,
        },
        "CTRLEN10": {
            "path": f"{top_path}/CTRLEN10_15day_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy/archive/post/atm/180x360_aave/ts/daily",
            "template": "PRECT.%(ensemble)s.%(year)s.nc",
            "frequency": "daily",
            "nens": 10,
        },
        "CAPTEN10": {
            "path": f"{top_path}/CAPTEN10_15day_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy/archive/post/atm/180x360_aave/ts/daily",
            "template": "PRECT.%(ensemble)s.%(year)s.nc",
            "frequency": "daily",
            "nens": 10,
        },
        "DARTEN20": {
            "path": f"{top_path}/DARTEN20_15day_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy/archive/post/atm/180x360_aave/ts/daily",
            "template": "PRECT.%(ensemble)s.%(year)s.nc",
            "frequency": "daily",
            "nens": 20,
        },
        "DARTEN40": {
            "path": f"{top_path}/DARTEN40_15day_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy/archive/post/atm/180x360_aave/ts/daily",
            "template": "PRECT.%(ensemble)s.%(year)s.nc",
            "frequency": "daily",
            "nens": 40,
        },
        "DARTEN40S1": {
            "path": f"{top_path}/DARTEN40S1_15day_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy/archive/post/atm/180x360_aave/ts/daily",
            "template": "PRECT.%(ensemble)s.%(year)s.nc",
            "frequency": "daily",
            "nens": 40,
        },
    }

    # --- Step 1: compute & save per (model, date) ---
    proc = SurfaceVarDiffProcessor(
        data_dict=data_dict,
        variable=variable,
        obs_variable=obs_variable,
        fscale=fscale,
        obs_fscale=obs_fscale,
        landmask_file=landmask_file,
        landmask_var=landmask_var,   # <- pass the string, not a bare name
        mask_land=mask_land,
        confidence_level=confidence_level,
    )

    for model_key in model_keys:
        if model_key in ["DARTEN40S1"]:
            target_dates = ["2012-06-01", "2012-06-05", "2012-06-10", "2012-06-15"]
        else:
            target_dates = ["2012-01-01", "2012-01-05", "2012-01-10", "2012-01-15"]
            
        for target_date in target_dates:
            outfile = os.path.join(
                out_path,
                f"{variable}_bias_spread_{model_key}_vs_{obs_key}_{target_date.replace('-','')}.nc",
            )
            try:
                proc.process_and_save(
                    model_key=model_key,
                    obs_key=obs_key,
                    target_date=target_date,
                    out_nc=outfile,
                )
            except FileNotFoundError as e:
                print(f"[SKIP] {model_key} {target_date}: {e}")
            except Exception as e:
                print(f"[ERROR] {model_key} {target_date}: {e}")
                

In [ ]:
if __name__ == "__main__":
    top_path = "/compyfs/zhan391/v3_dart_cda_scratch"
    out_path = "/compyfs/zhan391/v3_dart_cda_scratch/diag_dart"
    fig_path = "/qfs/people/zhan391/e3sm_dart_work/analysis/diagnostic/figures/mjo_analysis"
    rmm_dir = "/compyfs/zhan391/acme_init/Observations/MJO_INDEX"
    compset = "F20TR"
    resolution = "ne30pg2_r05_IcoswISC30E3r5"
    machine = "compy"
    exp_base = f"{compset}_{resolution}_{machine}"

    os.makedirs(out_path, exist_ok=True)
    os.makedirs(fig_path, exist_ok=True)

    region_dict = {"Tropics": [-10, 10]}
    
    varname = 'PRECT'
    var_dict = {
        'PRECT': {'alias':'Precipitation', 'units': "mm/day"} 
    }
    varname = "PRECT"
    vmin    = -5
    vmax    = 5 
    nlevs   = 11
    
    frequency = "daily"
    time_start = "2011-12-01"
    time_end = "2012-02-29"
    
    model_list = list(data_dict.keys())

    analyzer = HovmollerPrecipAnalyzer(
        data_dict=data_dict,
        varname=varname,
        var_info=var_dict[varname],
        lat_bounds=region_dict[region],
        dt_hours=24,
        rmm_path=rmm_dir,
        time_start = time_start, 
        time_end = time_end
    )

    analyzer.plot_rmm_phase_space(
        start=time_start, 
        end=time_end,
        amp_thresh=1.0,
        savepath=os.path.join(
            fig_path, 
            "rmm_phase_space.pdf"
        )
    )
    
    for exp in model_list:
        print(f"\n[INFO] Running MJO analysis for {exp}")
        output_dir = os.path.join(fig_path, f"mjo_{exp}")
        os.makedirs(output_dir, exist_ok=True)
        
        comps = analyzer.compute_phase_composites(exp)
        figname = os.path.join(output_dir, f"phase_composite_mjo_{exp}.pdf")
        analyzer.plot_phase_composites(comps,figname)
